# Notebook 10 — Correlaciones lineales y relevancia de las estadísticas

Este notebook explora las relaciones lineales entre las variables numéricas de las cuatro tablas históricas de La Liga. El objetivo es **describir asociaciones** y detectar métricas que merecen estudiarse como posibles variables predictoras; no se entrena ningún modelo de Machine Learning aquí.

> **Importante:** una correlación alta no demuestra causalidad ni garantiza que una variable mejore una predicción. Para hablar de impacto predictivo real hay que usar solamente información disponible antes del partido y validarla con una división temporal, como en el notebook 09.

## 1. Librerías y carga de datos

In [1]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)

ruta_clean = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "fbref", "processed", "clean"))
archivos = {
    "Partidos": "partidos_historico_limpio.csv",
    "Equipos": "equipos_historico_limpio.csv",
    "Tabla general": "tabla_general_historico_limpio.csv",
    "Jugadores": "jugadores_historico_limpio.csv",
}

datos = {
    nombre: pd.read_csv(os.path.join(ruta_clean, archivo), low_memory=False)
    for nombre, archivo in archivos.items()
}

for nombre, df in datos.items():
    print(f"{nombre}: {df.shape[0]:,} filas x {df.shape[1]} columnas")

Partidos: 7,720 filas x 8 columnas
Equipos: 420 filas x 22 columnas
Tabla general: 420 filas x 12 columnas
Jugadores: 11,765 filas x 24 columnas


## 2. Preparación de variables numéricas

Pearson mide asociación lineal entre dos variables numéricas. No incluimos identificadores como `Rk`, `Born` o `Temporada` en los mapas: aunque sean números, no representan una magnitud deportiva continua. Se exige además un mínimo de observaciones válidas para no interpretar correlaciones calculadas con muy pocos registros.

In [2]:
EXCLUIR_IDENTIFICADORES = {"Temporada", "Rk", "Born"}

def columnas_numericas_analizables(df):
    columnas = df.select_dtypes(include=np.number).columns.tolist()
    return [columna for columna in columnas if columna not in EXCLUIR_IDENTIFICADORES]

numericas = {
    nombre: df[columnas_numericas_analizables(df)].copy()
    for nombre, df in datos.items()
}

display(pd.DataFrame({
    "Tabla": list(numericas),
    "Variables analizadas": [len(df.columns) for df in numericas.values()],
    "Observaciones": [len(df) for df in numericas.values()],
}))

,Tabla,Variables analizadas,Observaciones
0,Partidos,4,7720
1,Equipos,20,420
2,Tabla general,9,420
3,Jugadores,18,11765


## 3. Mapas de calor de Pearson

La escala va de -1 a 1: valores cercanos a 1 indican asociación positiva, cercanos a -1 asociación negativa y cercanos a 0 poca relación lineal. La máscara muestra solo la mitad inferior porque la matriz es simétrica.

In [3]:
def graficar_correlaciones(df_numerico, titulo, figsize=(12, 9)):
    matriz = df_numerico.corr(method="pearson", min_periods=30)
    mask = np.triu(np.ones_like(matriz, dtype=bool))
    plt.figure(figsize=figsize)
    sns.heatmap(
        matriz, mask=mask, cmap="coolwarm", vmin=-1, vmax=1, center=0,
        annot=len(matriz.columns) <= 12, fmt=".2f", linewidths=0.4,
        cbar_kws={"label": "Coeficiente de Pearson"},
    )
    plt.title(f"Correlaciones lineales — {titulo}", fontweight="bold")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

for nombre, df_numerico in numericas.items():
    graficar_correlaciones(df_numerico, nombre, figsize=(14, 10))

: 

## 4. Ranking de relaciones más fuertes

Se muestran los pares sin duplicados y se conserva el signo original. El valor absoluto ordena la fuerza de la asociación, pero no debe confundirse con un efecto causal.

In [ ]:
def top_correlaciones(df_numerico, top_n=12, min_periods=30):
    matriz = df_numerico.corr(method="pearson", min_periods=min_periods)
    pares = []
    for i, columna_x in enumerate(matriz.columns):
        for columna_y in matriz.columns[i + 1:]:
            valor = matriz.loc[columna_x, columna_y]
            if pd.notna(valor):
                pares.append({
                    "Variable X": columna_x,
                    "Variable Y": columna_y,
                    "Pearson": valor,
                    "Fuerza absoluta": abs(valor),
                })
    return pd.DataFrame(pares).sort_values("Fuerza absoluta", ascending=False).head(top_n)

for nombre, df_numerico in numericas.items():
    print(f"\n--- {nombre} ---")
    display(top_correlaciones(df_numerico))

## 5. Relaciones con resultados de cada tabla

Una matriz completa puede destacar relaciones obvias o matemáticamente derivadas. Aquí ordenamos específicamente las correlaciones de cada variable con un resultado de referencia. Estas asociaciones son descriptivas y, cuando el resultado se calcula con la misma temporada, pueden contener información posterior al momento de predicción.

In [ ]:
objetivos = {
    "Equipos": "G+A_per90",
    "Tabla general": "Pts",
    "Jugadores": "G+A.1",
}

def correlaciones_con_objetivo(df, objetivo, min_periods=30):
    columnas = columnas_numericas_analizables(df)
    if objetivo not in df.columns or objetivo not in columnas:
        return pd.DataFrame(columns=["Variable", "Pearson", "n"])
    filas = []
    for variable in columnas:
        if variable == objetivo:
            continue
        pares = df[[variable, objetivo]].dropna()
        if len(pares) >= min_periods:
            filas.append({"Variable": variable, "Pearson": pares[variable].corr(pares[objetivo]), "n": len(pares)})
    return pd.DataFrame(filas).sort_values("Pearson", key=lambda serie: serie.abs(), ascending=False)

for nombre, objetivo in objetivos.items():
    print(f"\n{nombre} — objetivo descriptivo: {objetivo}")
    display(correlaciones_con_objetivo(datos[nombre], objetivo).head(10))

## 6. Gráficas de dispersión y línea de tendencia

La línea roja es el ajuste lineal de mínimos cuadrados. Se seleccionan automáticamente las tres relaciones más fuertes de cada tabla; cambiar `tabla_graficas` permite inspeccionar otra tabla.

In [ ]:
def graficar_regresiones(df, tabla_graficas, top_n=3):
    pares = top_correlaciones(numericas[tabla_graficas], top_n=top_n)
    for _, par in pares.iterrows():
        x, y = par["Variable X"], par["Variable Y"]
        plt.figure(figsize=(7, 5))
        sns.regplot(
            data=df, x=x, y=y, ci=95,
            scatter_kws={"alpha": 0.45, "color": "#001b36"},
            line_kws={"color": "red", "linewidth": 2},
        )
        plt.title(f"{tabla_graficas}: {x} vs {y}\nPearson = {par['Pearson']:.3f}")
        plt.tight_layout()
        plt.show()

graficar_regresiones(datos["Equipos"], "Equipos")
# Ejemplos: graficar_regresiones(datos["Jugadores"], "Jugadores")
#           graficar_regresiones(datos["Tabla general"], "Tabla general")

## 7. Correlación sin fuga: forma previa y resultado del partido

Esta es la parte más cercana a la pregunta predictiva. Para cada partido calculamos el promedio de puntos y de diferencia de goles de los **cinco partidos anteriores** de cada equipo. Las variables se construyen antes de leer el resultado del partido actual, por lo que no usan `Goles_Local`, `Goles_Visita` ni puntos del propio partido.

Esto sigue siendo un análisis bivariado, no una prueba de que una métrica cause el resultado ni una evaluación completa de un modelo.

In [ ]:
def construir_forma_previa(partidos, ventana=5):
    requeridas = {"Temporada", "Date", "Home", "Away", "Goles_Local", "Goles_Visita", "Puntos_Local", "Puntos_Visita"}
    faltantes = requeridas - set(partidos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {sorted(faltantes)}")

    base = partidos.copy()
    base["_fecha"] = pd.to_datetime(base["Date"], errors="coerce")
    if base["_fecha"].isna().any():
        raise ValueError("La columna Date contiene fechas invalidas.")
    base = base.sort_values(["_fecha", "Home", "Away"]).reset_index(drop=True)
    historial = {}
    registros = []

    def promedio(equipo, metrica):
        valores = historial.get(equipo, {}).get(metrica, [])[-ventana:]
        return float(np.mean(valores)) if valores else np.nan

    for _, partido in base.iterrows():
        local, visitante = partido["Home"], partido["Away"]
        registro = {
            "Home_PtsPrev5": promedio(local, "puntos"),
            "Away_PtsPrev5": promedio(visitante, "puntos"),
            "Home_GDPrev5": promedio(local, "dif_goles"),
            "Away_GDPrev5": promedio(visitante, "dif_goles"),
            "Resultado_Local_Puntos": partido["Puntos_Local"],
            "Diferencia_Goles": partido["Goles_Local"] - partido["Goles_Visita"],
        }
        registros.append(registro)
        historial.setdefault(local, {"puntos": [], "dif_goles": []})
        historial.setdefault(visitante, {"puntos": [], "dif_goles": []})
        historial[local]["puntos"].append(partido["Puntos_Local"])
        historial[local]["dif_goles"].append(partido["Goles_Local"] - partido["Goles_Visita"])
        historial[visitante]["puntos"].append(partido["Puntos_Visita"])
        historial[visitante]["dif_goles"].append(partido["Goles_Visita"] - partido["Goles_Local"])

    return pd.DataFrame(registros).dropna()

forma_previa = construir_forma_previa(datos["Partidos"])
display(forma_previa.head())
display(forma_previa.corr(method="pearson").loc[["Resultado_Local_Puntos", "Diferencia_Goles"]])

In [ ]:
matriz_previa = forma_previa.corr(method="pearson")
plt.figure(figsize=(9, 6))
sns.heatmap(matriz_previa, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0)
plt.title("Forma previa y resultado del partido (sin fuga de información)")
plt.tight_layout()
plt.show()

for objetivo in ["Resultado_Local_Puntos", "Diferencia_Goles"]:
    ranking = forma_previa.drop(columns=[objetivo]).corrwith(forma_previa[objetivo]).sort_values(key=lambda s: s.abs(), ascending=False)
    print(f"Correlaciones con {objetivo}")
    display(ranking.to_frame("Pearson"))

## 8. Conclusiones y límites

- Las correlaciones altas sirven para priorizar métricas y detectar redundancia entre columnas.
- Las relaciones de una tabla con `Pts`, goles o clasificación pueden ser contemporáneas o posteriores al resultado; no deben presentarse como predictores válidos sin una fecha de corte.
- La sección de forma previa es una evidencia descriptiva más honesta para predicción porque usa únicamente partidos anteriores.
- Para afirmar que una métrica tiene impacto real en las predicciones, el siguiente paso es comparar un modelo base contra otro que la incluya, usando validación temporal y métricas como *log loss*, *F1* o *Brier score*. La correlación por sí sola no prueba ese impacto.